# Predire la potenza a T_Lac con la regressione Ridge

L'obiettivo è stimare la potenza associata a T_Lac, espressa in watt (`P_TLac_W`), a partire da misure antropometriche, ematologiche e cardiorespiratorie non derivate dal lattato. Il target corrisponde al valore «Load @ Tlac» riportato nei file originali; non viene ricalcolato dalle curve di lattato.

Il lavoro comprende l'importazione dei test ciclistici, il controllo della qualità dei dati, la costruzione delle covariate e la valutazione di una regressione Ridge. La procedura è retrospettiva ed esplorativa: le stime non hanno validità clinica dimostrata e T_Lac non viene equiparato a IAT, LT1, LT2 o MLSS.

## 1. Preparare l'ambiente

Eseguire i comandi nel terminale PowerShell dalla cartella principale del repository. L'ambiente utilizzato è Python 3.14; le versioni delle librerie sono specificate in `requirements.txt`.

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python -m pytest -q
```

Il notebook si può leggere in VS Code o Jupyter senza eseguire celle Python. Tutti i passaggi operativi sono descritti attraverso comandi da terminale.

## 2. Importare i dati e verificarne l'integrità

I dati provengono dall'archivio `data_v2.zip` di Donne et al., disponibile su [Zenodo](https://zenodo.org/records/10841412). Per una nuova copia del progetto, scaricare l'archivio nella cartella degli originali:

```powershell
New-Item -ItemType Directory -Force data/raw
Invoke-WebRequest -Uri "https://zenodo.org/records/10841412/files/data_v2.zip?download=1" -OutFile data/raw/data_v2.zip
python -m src.build_report --raw data/raw --output reports/data
```

Se l'archivio è già presente, eseguire soltanto l'ultimo comando. La procedura verifica il checksum MD5 `f64cb1bf4d66129daed6c1954ea6ca9f` e l'integrità dello ZIP, estrae gli Excel e seleziona le voci «Cycling» dell'indice `Data_Summary.xlsx`. Gli originali vengono conservati senza modifiche.

Si ottengono il manifest `data/processed/cycling.json`, l'audit `reports/data/data_audit.md` e una pagina esplorativa in `reports/data/index.html`. Prima di proseguire, consultare l'audit per controllare target mancanti, unità e anomalie. I dati grezzi e quelli preparati restano locali e sono esclusi da Git.

## 3. Costruire il dataset

```powershell
python -m src.build_features --manifest data/processed/cycling.json --output data/processed --reports reports/data
```

Ogni riga rappresenta un test ciclistico. Dei 285 test disponibili, A229 e A519 vengono esclusi perché il target è mancante o non finito: rimangono 283 osservazioni. Il comando produce `features_unimputed.csv`, `features_manifest.json` ed `excluded_tests.csv` in `data/processed/`, insieme al report `reports/data/features.md`.

Il target non viene imputato. I valori mancanti delle covariate restano nel CSV e vengono trattati durante l'addestramento. Le sintesi dei segnali, come frequenza cardiaca e VO₂ mediani, sono calcolate sugli step con potenza positiva. L'altezza è espressa in metri, la massa in kg, la potenza in W e la frequenza cardiaca in bpm; le unità ematologiche non sono dichiarate nella fonte e vengono mantenute nella scala originale.

## 4. Preparare train, test e fold di validazione

La divisione iniziale è 80/20, con seed 42 e stratificazione per sesso e fasce di età (<25, 25–34, ≥35 anni, età mancante). Produce 226 osservazioni di train e 57 di test. Per crearla in una nuova cartella di lavoro:

```powershell
python -m src.split_dataset --config configs/split.json
```

Il comando salva `train.csv`, `test.csv`, `cv_folds.csv`, `train_manifest.json` e `split_manifest.json` in `data/splits/v1/`. Se questa cartella esiste già, utilizzare la partizione presente: il comando non la sovrascrive. Gli identificativi distinguono i test, ma non consentono di identificare stabilmente gli atleti; non è quindi garantita la separazione di eventuali test ripetuti della stessa persona.

Per il modello finale sono state escluse dal solo train quattro osservazioni:

| Test | Motivazione |
|---|---|
| A539 | Incoerenza fra VO₂max assoluto, VO₂max relativo e massa |
| A1114 | VO₂ a 160 bpm sospetto; errore non confermato |
| A4000 | Target di 266 W superiore alla potenza massima osservata di 240 W |
| A4001 | Target di 266 W superiore alla potenza massima osservata di 250 W |

Il train risultante contiene 222 test; le assegnazioni originali ai cinque fold sono conservate, con 45, 45, 43, 44 e 45 osservazioni di validazione. Il test resta composto da 57 osservazioni. Queste esclusioni sono scelte esplorative di controllo qualità, non una regola automatica applicabile a nuovi dati.

**Per riprodurre i risultati riportati occorre importare anche la cartella preparata `v2_train_qc`.** Il repository non contiene un comando CLI che ricostruisca queste quattro esclusioni e il relativo manifest. Servono `train.csv`, `cv_folds.csv`, `train_manifest.json` ed `exclusions.csv`, conservati insieme ai dati locali dello studio. Per importarli da una copia completa dei dati, sostituire il percorso di esempio:

```powershell
$origineDati = 'C:\percorso\copia-dati'
Copy-Item -LiteralPath "$origineDati\splits\v2_train_qc" -Destination data/splits/v2_train_qc -Recurse
```

Eseguire la copia solo se la destinazione non esiste. Se si dispone già anche dello split originale, importare allo stesso modo `splits/v1` invece di rigenerarlo. Senza i file del train QC si può ripetere l'importazione e l'audit, ma non riprodurre esattamente la Ridge descritta qui.

## 5. Definire le covariate e le ipotesi del modello

La configurazione `configs/first_model/training_ridge.json` specifica le 20 covariate utilizzate:

| Gruppo | Colonne |
|---|---|
| Sesso | `gender` |
| Antropometria | `height_m`, `mass_kg`, `bmi_kg_m2`, `skinfolds_mm`, `fat_pct`, `lbm_kg` |
| Ematologia | `hb`, `hct`, `rbc`, `wbc` |
| Misure cardiorespiratorie | `vo2max_ml_kg_min`, `vo2max_l_min`, `ve_max_l_min`, `hr_max_bpm`, `vo2_160bpm_ml_kg_min`, `power_160bpm_W` |
| Età e sintesi del test | `age_years`, `median_hr_bpm`, `median_vo2_ml_kg_min` |

Non sono incluse la potenza massima, il numero di step, la cadenza o variabili derivate dal lattato. Si usa un unico insieme di 20 covariate, senza confronto A/B, PCA o selezione progressiva delle feature. Alcuni input richiedono un test da sforzo: il modello non stima la soglia a partire dalle sole misure a riposo.

La Ridge descrive il target come combinazione lineare delle covariate, penalizzando l'ampiezza dei coefficienti. I mancanti numerici vengono imputati con mediane per sesso; il sesso è codificato F=0 e M=1 e tutte le covariate sono standardizzate. Imputazione e standardizzazione vengono stimate esclusivamente sui dati di addestramento di ciascun fit. La regolarizzazione attenua l'instabilità legata alla correlazione fra variabili, senza rendere causale l'interpretazione dei coefficienti.

## 6. Addestrare la Ridge

```powershell
python -m src.first_model.train --config configs/first_model/training_ridge.json --model ridge --output results/runs/ridge_reproduction --refit
```

La validazione incrociata è annidata: cinque fold esterni stimano le prestazioni e quattro fold interni selezionano `alpha` minimizzando il MAE. Il seed è 42. La ricerca considera dieci valori: 0,0001; 0,001; 0,01; 0,1; 1; 10; 100; 1000; 10000; 100000.

Ogni predizione out-of-fold (OOF) è ottenuta da un modello addestrato senza quella osservazione. Con `--refit`, la ricerca viene poi ripetuta sul train completo di 222 test e il modello selezionato viene adattato a tutte queste osservazioni. Nella run di riferimento, `results/runs/ridge_final/`, l'alpha finale è **0,1**.

La nuova run salva il riepilogo della validazione, le predizioni OOF e il modello con il suo preprocessing in `results/runs/ridge_reproduction/`. Usare una destinazione nuova per ogni ripetizione: le run esistenti non vengono sovrascritte.

## 7. Valutare il modello sul test

```powershell
python -m src.first_model.evaluate --run results/runs/ridge_reproduction --holdout-dir data/splits/v1 --output results/test_ridge_reproduction --allow-test-access
```

La valutazione applica ai 57 test il preprocessing e la Ridge salvati, senza adattarli nuovamente. Il comando controlla la provenienza dei file e la disgiunzione degli identificativi e produce predizioni, metriche e riepilogo con intervalli bootstrap nella cartella indicata.

Gli intervalli al 95% sono calcolati con 2000 ricampionamenti delle righe del test e seed 42, mantenendo fisso il modello. Descrivono la variabilità delle metriche sul campione disponibile; non sono intervalli di previsione individuali e non includono l'incertezza dell'addestramento.

## 8. Risultati della Ridge

| Metrica | Validazione OOF (222 test) | Test (57 test) | Intervallo bootstrap 95% sul test |
|---|---:|---:|---:|
| MAE (W) | 15,93 | 12,69 | 10,06–15,63 |
| RMSE (W) | 19,89 | 16,57 | 12,63–20,77 |
| R² | 0,810 | 0,867 | 0,751–0,928 |
| Bias, predetto − osservato (W) | +0,35 | −3,81 | — |

Il MAE sul test corrisponde a uno scarto assoluto medio di circa 13 W. Il RMSE più elevato indica la presenza di alcuni errori maggiori; il bias negativo segnala una lieve sottostima media. L'errore assoluto mediano è 11,18 W e il 90° percentile è 25,34 W.

Le prestazioni sul test sono numericamente migliori di quelle OOF, ma le due stime derivano da campioni e numerosità di addestramento diversi. La differenza non dimostra un miglioramento intrinseco del modello.

![Prestazioni della Ridge](reports/ridge/performance.png)

Le metriche sono disponibili in [results/metrics.csv](results/metrics.csv); gli intervalli e le predizioni della valutazione di riferimento sono conservati in [results/test_ridge/summary.json](results/test_ridge/summary.json) e [results/test_ridge/predictions.csv](results/test_ridge/predictions.csv).

## 9. Interpretazione e limiti

Fra i coefficienti standardizzati di maggiore ampiezza figurano la potenza a 160 bpm (+47,06 W per deviazione standard), la massa (+36,81 W), la frequenza cardiaca mediana (+21,93 W), la massa magra (−19,64 W) e il VO₂max relativo (+18,86 W). I coefficienti descrivono associazioni condizionate alle altre covariate: segni opposti fra misure correlate, come massa e massa magra, non implicano effetti fisiologici opposti.

![Coefficienti della Ridge](reports/ridge/coefficients.png)

I [coefficienti e le scale](reports/ridge/coefficients.csv) consentono di consultare la parametrizzazione del modello. Per applicarlo a nuovi test servono le stesse 20 covariate, nelle stesse unità, e il preprocessing salvato insieme alla Ridge; non occorre misurare il target.

La numerosità contenuta, la provenienza da un'unica fonte, la possibile presenza di test ripetuti e le esclusioni esplorative limitano la generalizzabilità. Nel test sette valori mancanti sono stati imputati usando il train. La collinearità limita inoltre la stabilità dei singoli coefficienti. Una valutazione su nuovi atleti e dati indipendenti è necessaria per stabilire quanto queste prestazioni siano trasferibili.